# Notebook 06 — Calibration, Explainability, and Decision Support

## Objective

This notebook completes the final analytical stage of the multi-modal credit-risk
study.

Using the model results established in the preceding notebooks, it evaluates the
selected hybrid model from three perspectives:

1. probability calibration and reliability;
2. model explainability; and
3. translation of predicted default probabilities into a simple risk-based
   decision-support framework.

The analysis is restricted to the previously defined temporal train, validation,
and held-out 2014 test cohorts. No additional predictive model families are
introduced in this notebook.

The objective is to determine whether the selected model produces outputs that
are sufficiently interpretable and probabilistically meaningful to support
transparent Approve, Review, and Reject risk categories.

## 1. Environment and Artifact Setup

This section initializes the notebook environment, defines the project paths,
and verifies that the required outputs from Notebook 05 are available before
the final evaluation begins.


In [24]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Credit_Risk_Thesis"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

NB5_OUTPUT_DIR = (
    PROCESSED_DATA_DIR / "notebook_05"
)

NB6_OUTPUT_DIR = (
    PROCESSED_DATA_DIR / "notebook_06"
)

NB6_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Notebook 05:", NB5_OUTPUT_DIR)
print("Notebook 06:", NB6_OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook 05: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_05
Notebook 06: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_06


### 1.1 Verify Notebook 05 Handoff

The final evaluation depends on persisted comparison and calibration artifacts
from Notebook 05. Their availability is verified before loading any results.


In [25]:
required_nb5_files = [
    "final_representation_comparison.csv",
    "matched_representation_calibration.csv",
    "representation_paired_bootstrap_comparison.csv"
]

for filename in required_nb5_files:

    path = NB5_OUTPUT_DIR / filename

    print(
        filename,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

final_representation_comparison.csv -> FOUND
matched_representation_calibration.csv -> FOUND
representation_paired_bootstrap_comparison.csv -> FOUND


## 2. Final Model Selection

The representation experiments in Notebook 05 compared the structured-only
baseline with traditional TF-IDF and contextual BERT text-enhanced models on
the same matched borrower cohort.

The Structured + BERT model achieved the highest test ROC-AUC (0.6852), while
Structured + TF-IDF achieved a slightly lower ROC-AUC (0.6834) but the highest
PR-AUC (0.2788), lowest Brier score (0.1261), and lower Expected Calibration
Error (0.0089).

The paired bootstrap comparison did not provide clear evidence that the small
ROC-AUC difference between the two hybrid models was stable under resampling.

Because the subsequent analysis depends on reliable probability estimates and
transparent model interpretation, the **Structured + TF-IDF Logistic Regression**
model is selected for the final calibration, explainability, and decision-support
analysis.

The BERT hybrid remains an important comparative result but is not treated as
the final operational research model.

In [26]:
representation_comparison = pd.read_csv(
    NB5_OUTPUT_DIR / "final_representation_comparison.csv"
)

calibration_comparison = pd.read_csv(
    NB5_OUTPUT_DIR / "matched_representation_calibration.csv"
)

bootstrap_comparison = pd.read_csv(
    NB5_OUTPUT_DIR
    / "representation_paired_bootstrap_comparison.csv"
)

print("Final representation comparison:")
display(representation_comparison)

print("\nCalibration comparison:")
display(calibration_comparison)

Final representation comparison:


,representation,roc_auc,pr_auc,brier_score
0,TF-IDF text-only,0.571466,0.194471,0.132339
1,BERT text-only,0.585693,0.201401,0.132398
2,Structured-only,0.678005,0.275648,0.126489
3,Structured + TF-IDF,0.683357,0.278794,0.126090
4,Structured + BERT,0.685176,0.276257,0.126880



Calibration comparison:


,representation,brier_score,ece_10_bins,mean_predicted_probability,observed_default_rate
0,TF-IDF text-only,0.1323,0.0077,0.1508,0.1585
1,BERT text-only,0.1324,0.0166,0.1742,0.1585
2,Structured-only,0.1265,0.0130,0.1685,0.1585
3,Structured + TF-IDF,0.1261,0.0089,0.1639,0.1585
4,Structured + BERT,0.1269,0.0276,0.1861,0.1585


In [27]:
SELECTED_MODEL = "Structured + TF-IDF"

selected_representation = representation_comparison[
    representation_comparison["representation"] == SELECTED_MODEL
].iloc[0]

selected_calibration_row = calibration_comparison[
    calibration_comparison["representation"] == SELECTED_MODEL
].iloc[0]

selected_model_summary = pd.DataFrame([{
    "selected_model": SELECTED_MODEL,
    "roc_auc": selected_representation["roc_auc"],
    "pr_auc": selected_representation["pr_auc"],
    "brier_score": selected_representation["brier_score"],
    "ece_10_bins": selected_calibration_row["ece_10_bins"],
    "selection_reason":
        "Best overall balance of discrimination, "
        "probability reliability, and interpretability"
}])

selected_model_summary.round(4)


,selected_model,roc_auc,pr_auc,brier_score,ece_10_bins,selection_reason
0,Structured + TF-IDF,0.6834,0.2788,0.1261,0.0089,"Best overall balance of discrimination, probab..."


In [28]:
selected_model_summary.to_csv(
    NB6_OUTPUT_DIR / "model_selection_summary.csv",
    index=False
)

print("Model selection summary saved.")


Model selection summary saved.


## 3. Calibration Assessment

Credit-risk decision support requires predicted probabilities to be meaningful
as estimates of default risk. Therefore, the selected Structured + TF-IDF model
is assessed for probability calibration before its outputs are used to construct
risk-based decision categories.

Calibration is evaluated using the Brier score, Expected Calibration Error (ECE),
and the relationship between mean predicted and observed default probabilities.

Post-hoc recalibration is considered only if the existing model shows meaningful
evidence of probability miscalibration.

### 3.1 Selected-Model Calibration Metrics

Calibration results already computed on the matched held-out test cohort are
loaded from Notebook 05 for the selected Structured + TF-IDF model.


In [29]:
selected_calibration = calibration_comparison[
    calibration_comparison["representation"]
    == "Structured + TF-IDF"
].copy()

selected_calibration

,representation,brier_score,ece_10_bins,mean_predicted_probability,observed_default_rate
3,Structured + TF-IDF,0.1261,0.0089,0.1639,0.1585


### 3.2 Predicted and Observed Default Risk

The aggregate mean predicted probability is compared with the observed default
rate as an additional descriptive check of probability reliability.


In [30]:
mean_pred = selected_calibration[
    "mean_predicted_probability"
].iloc[0]

observed_rate = selected_calibration[
    "observed_default_rate"
].iloc[0]

probability_gap = mean_pred - observed_rate

print(
    "Mean predicted default probability:",
    round(mean_pred, 4)
)

print(
    "Observed default rate:",
    round(observed_rate, 4)
)

print(
    "Difference:",
    round(probability_gap, 4)
)

print(
    "Absolute difference:",
    round(abs(probability_gap), 4)
)

Mean predicted default probability: 0.1639
Observed default rate: 0.1585
Difference: 0.0054
Absolute difference: 0.0054


### 3.3 Calibration Conclusion

The selected Structured + TF-IDF model produced a mean predicted default
probability of 0.1639 on the held-out 2014 test cohort, compared with an
observed default rate of 0.1585. The absolute difference was 0.0054, equivalent
to approximately 0.54 percentage points.

The model achieved a Brier score of 0.1261 and a 10-bin Expected Calibration
Error (ECE) of 0.0089. These results indicate relatively small probability
discrepancies under the calibration measures used in this study.

Although the model slightly overestimated aggregate default risk, the observed
difference was small. Therefore, no additional post-hoc recalibration was
applied. The original predicted probabilities are retained for the subsequent
risk-based decision-support analysis.


## 4. Explainability Handoff

Explainability of the Logistic Regression models was examined during the
representation-modelling stage in Notebook 05. Coefficients were used to
identify structured and textual features associated with higher and lower
predicted default risk, with interpretation restricted to predictive
associations rather than causal effects.

The purpose of this section is therefore not to repeat the coefficient analysis,
but to verify that the exact selected model, preprocessing objects, and ordered
feature definitions are available for reproducibility and subsequent reporting.


### 4.1 Load Persisted Model Components

The selected Structured + TF-IDF model, fitted preprocessing objects, and saved
test probabilities are loaded from Notebook 05 rather than reconstructed.


In [31]:
import joblib
import numpy as np

selected_hybrid_model = joblib.load(
    NB5_OUTPUT_DIR / "structured_tfidf_logistic_model.joblib"
)

tfidf_vectorizer = joblib.load(
    NB5_OUTPUT_DIR / "tfidf_vectorizer.joblib"
)

matched_encoder = joblib.load(
    NB5_OUTPUT_DIR / "matched_categorical_encoder.joblib"
)

matched_numeric_scaler = joblib.load(
    NB5_OUTPUT_DIR / "matched_numeric_scaler.joblib"
)

selected_test_prob = np.load(
    NB5_OUTPUT_DIR / "structured_tfidf_test_prob.npy"
)

print("Selected hybrid model loaded.")
print("Model coefficient count:", selected_hybrid_model.coef_.shape[1])
print("Saved test probabilities:", selected_test_prob.shape)

Selected hybrid model loaded.
Model coefficient count: 42524
Saved test probabilities: (15175,)


### 4.2 Reconstruct Ordered Feature Definitions

Categorical one-hot feature names, TF-IDF vocabulary terms, and the persisted
numerical feature list are combined in the same order used when fitting the
selected hybrid model.


In [32]:
categorical_encoded_names = (
    matched_encoder.get_feature_names_out()
)

tfidf_feature_names = (
    tfidf_vectorizer.get_feature_names_out()
)

print(
    "Categorical encoded features:",
    len(categorical_encoded_names)
)

print(
    "TF-IDF features:",
    len(tfidf_feature_names)
)

Categorical encoded features: 25
TF-IDF features: 42436


In [33]:
import json

with open(
    NB5_OUTPUT_DIR / "matched_numerical_feature_names.json",
    "r"
) as f:
    numerical_feature_names = json.load(f)

print(
    "Numerical structured features:",
    len(numerical_feature_names)
)

Numerical structured features: 63


### 4.3 Validate Feature/Model Alignment

The combined feature-name sequence must exactly match the fitted Logistic
Regression coefficient vector before the model is treated as reproducible and
interpretable.


In [34]:
structured_feature_names = np.concatenate([
    np.array(numerical_feature_names),
    categorical_encoded_names
])

hybrid_feature_names = np.concatenate([
    structured_feature_names,
    tfidf_feature_names
])

print(
    "Structured feature names:",
    len(structured_feature_names)
)

print(
    "Hybrid feature names:",
    len(hybrid_feature_names)
)

print(
    "Model coefficients:",
    selected_hybrid_model.coef_.shape[1]
)

assert (
    len(hybrid_feature_names)
    == selected_hybrid_model.coef_.shape[1]
)

print("Feature-name alignment validated.")

Structured feature names: 88
Hybrid feature names: 42524
Model coefficients: 42524
Feature-name alignment validated.


### 4.4 Explainability Conclusion

Coefficient-based interpretation was completed during the representation and
hybrid-modelling stage and is therefore not duplicated here. The persisted
model components and ordered feature definitions confirm that the final
Structured + TF-IDF specification can be reproduced for dissertation reporting.

All coefficient interpretations are treated as predictive associations and not
as evidence of causal relationships.


## 5. Risk-Based Decision-Support Layer

The final stage translates predicted default probabilities into an interpretable
risk-based decision-support framework.

Rather than treating the conventional 0.50 classification threshold as a lending
decision rule, borrowers are grouped into three risk bands representing lower,
intermediate, and higher predicted default risk.

Decision thresholds are determined using the validation cohort rather than the
held-out test cohort. The resulting thresholds are then fixed and applied to the
2014 test cohort to evaluate the distribution and observed default behaviour
within each risk band.

The resulting categories are intended as a research-oriented decision-support
illustration rather than as an operational lending policy.

### 5.1 Load Validation and Test Probabilities

The selected model probabilities persisted in Notebook 05 are loaded so that
decision thresholds can be derived from validation data and subsequently
evaluated on the held-out test cohort.


In [35]:
selected_val_prob = np.load(
    NB5_OUTPUT_DIR / "structured_tfidf_val_prob.npy"
)

selected_test_prob = np.load(
    NB5_OUTPUT_DIR / "structured_tfidf_test_prob.npy"
)

print("Validation probabilities:", selected_val_prob.shape)
print("Test probabilities:", selected_test_prob.shape)

print(
    "\nValidation probability range:",
    round(selected_val_prob.min(), 4),
    "to",
    round(selected_val_prob.max(), 4)
)

print(
    "Validation probability mean:",
    round(selected_val_prob.mean(), 4)
)

Validation probabilities: (48726,)
Test probabilities: (15175,)

Validation probability range: 0.0 to 0.7948
Validation probability mean: 0.1615


### 5.2 Validation Probability Distribution

The validation probability distribution is inspected to support a transparent
choice of risk-band thresholds without using the held-out test outcomes.


In [36]:
probability_summary = pd.Series(
    selected_val_prob
).describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95
    ]
)

probability_summary

,0
count,4.872600e+04
mean,1.615357e-01
std,1.055376e-01
min,2.592210e-18
10%,5.122411e-02
25%,8.532696e-02
50%,1.371757e-01
75%,2.124526e-01
90%,3.072618e-01
95%,3.741545e-01


### 5.3 Define Validation-Derived Risk Thresholds

The median validation probability defines the lower/intermediate risk boundary,
while the 90th percentile defines the intermediate/higher risk boundary. These
thresholds are intended for research-oriented risk stratification rather than
as optimized operational lending rules.


In [37]:
LOW_RISK_THRESHOLD = np.quantile(
    selected_val_prob,
    0.50
)

HIGH_RISK_THRESHOLD = np.quantile(
    selected_val_prob,
    0.90
)

print(
    "Approve / Review threshold:",
    round(LOW_RISK_THRESHOLD, 4)
)

print(
    "Review / Reject threshold:",
    round(HIGH_RISK_THRESHOLD, 4)
)

Approve / Review threshold: 0.1372
Review / Reject threshold: 0.3073


In [38]:
def assign_risk_decision(probabilities):
    return np.where(
        probabilities < LOW_RISK_THRESHOLD,
        "Approve",
        np.where(
            probabilities < HIGH_RISK_THRESHOLD,
            "Review",
            "Reject"
        )
    )

### 5.4 Apply Fixed Thresholds to the Test Cohort

The validation-derived thresholds are applied unchanged to the temporally later
2014 test cohort. The held-out outcomes are used only to evaluate the resulting
risk stratification.


In [39]:
y_text_test = np.load(
    NB5_OUTPUT_DIR / "y_text_test.npy"
)

print("Test target:", y_text_test.shape)
print("Defaults:", int(y_text_test.sum()))
print(
    "Default rate:",
    round(y_text_test.mean(), 4)
)

assert len(y_text_test) == len(selected_test_prob)

print("Test target and probabilities aligned.")

Test target: (15175,)
Defaults: 2405
Default rate: 0.1585
Test target and probabilities aligned.


In [40]:
test_decisions = assign_risk_decision(
    selected_test_prob
)

decision_results = pd.DataFrame({
    "predicted_default_probability":
        selected_test_prob,
    "observed_default":
        y_text_test,
    "decision":
        test_decisions
})

decision_results.head()

,predicted_default_probability,observed_default,decision
0,0.100015,0,Approve
1,0.069958,0,Approve
2,0.181942,1,Review
3,0.165403,0,Review
4,0.130326,0,Approve


### 5.5 Evaluate Risk Stratification

Observed default rates and mean predicted probabilities are summarized within
the three resulting risk bands.


In [41]:
decision_summary = (
    decision_results
    .groupby("decision")
    .agg(
        borrowers=("observed_default", "size"),
        observed_defaults=("observed_default", "sum"),
        observed_default_rate=("observed_default", "mean"),
        mean_predicted_probability=(
            "predicted_default_probability",
            "mean"
        )
    )
    .reset_index()
)

decision_summary["borrower_pct"] = (
    decision_summary["borrowers"]
    / len(decision_results)
    * 100
)

decision_summary = decision_summary[
    [
        "decision",
        "borrowers",
        "borrower_pct",
        "mean_predicted_probability",
        "observed_default_rate",
        "observed_defaults"
    ]
]

decision_summary.round(4)

,decision,borrowers,borrower_pct,mean_predicted_probability,observed_default_rate,observed_defaults
0,Approve,7228,47.6310,0.0821,0.0834,603
1,Reject,1488,9.8056,0.3886,0.3421,509
2,Review,6459,42.5634,0.2036,0.2002,1293


### 5.6 Decision-Support Interpretation

The validation-derived thresholds produced clear risk stratification on the
held-out 2014 test cohort. The observed default rate increased from 8.34% in
the Approve group to 20.02% in the Review group and 34.21% in the Reject
group. Mean predicted default probabilities showed the same monotonic pattern
(8.21%, 20.36%, and 38.86%, respectively).

The thresholds were determined from the validation cohort and applied unchanged
to the temporally later test cohort. Therefore, the observed separation provides
evidence that the selected model can support meaningful risk stratification
under temporal holdout.

These categories are illustrative decision-support bands rather than optimized
or operational lending rules. No claim is made that the selected thresholds
represent economically optimal approval or rejection policies.


## 6. Ablation and Final Model Synthesis

An ablation-style comparison is used to assess the incremental contribution of
textual information to structured credit-risk prediction. Rather than fitting
additional models, this section consolidates the held-out test results obtained
from the representation experiments.

The structured-only model provides the reference baseline. Structured + TF-IDF
and Structured + BERT are then compared against this baseline to determine
whether the addition of borrower text improves discrimination and probabilistic
prediction.

Text-only TF-IDF and BERT results are retained as supplementary evidence of the
standalone predictive value of borrower descriptions.

### 6.1 Representation Comparison

Previously generated held-out test results are reused to compare structured-only,
text-only, and hybrid representations under the same matched cohort.


In [42]:
ablation_results = pd.read_csv(
    NB5_OUTPUT_DIR /
    "final_representation_comparison.csv"
)

ablation_results

,representation,roc_auc,pr_auc,brier_score
0,TF-IDF text-only,0.571466,0.194471,0.132339
1,BERT text-only,0.585693,0.201401,0.132398
2,Structured-only,0.678005,0.275648,0.126489
3,Structured + TF-IDF,0.683357,0.278794,0.126090
4,Structured + BERT,0.685176,0.276257,0.126880


### 6.2 Incremental Value over the Structured Baseline

The changes in ROC-AUC, PR-AUC, and Brier score are calculated relative to the
structured-only model to quantify the incremental contribution of text.


In [43]:
structured_baseline = ablation_results[
    ablation_results["representation"]
    == "Structured-only"
].iloc[0]

hybrid_ablation = ablation_results[
    ablation_results["representation"].isin([
        "Structured + TF-IDF",
        "Structured + BERT"
    ])
].copy()

hybrid_ablation["delta_roc_auc"] = (
    hybrid_ablation["roc_auc"]
    - structured_baseline["roc_auc"]
)

hybrid_ablation["delta_pr_auc"] = (
    hybrid_ablation["pr_auc"]
    - structured_baseline["pr_auc"]
)

# Lower Brier score is better, so positive improvement means
# the hybrid model reduced Brier score.
hybrid_ablation["brier_improvement"] = (
    structured_baseline["brier_score"]
    - hybrid_ablation["brier_score"]
)

hybrid_ablation[
    [
        "representation",
        "roc_auc",
        "delta_roc_auc",
        "pr_auc",
        "delta_pr_auc",
        "brier_score",
        "brier_improvement"
    ]
].round(4)

,representation,roc_auc,delta_roc_auc,pr_auc,delta_pr_auc,brier_score,brier_improvement
3,Structured + TF-IDF,0.6834,0.0054,0.2788,0.0031,0.1261,0.0004
4,Structured + BERT,0.6852,0.0072,0.2763,0.0006,0.1269,-0.0004


### 6.3 Ablation Interpretation

The ablation analysis indicates that borrower text provides modest incremental
predictive information beyond the structured credit variables.

Adding TF-IDF features to the structured baseline increased ROC-AUC from 0.6780
to 0.6834 and PR-AUC from 0.2756 to 0.2788, while reducing the Brier score from
0.1265 to 0.1261. This indicates a small improvement in both discrimination and
probability accuracy.

The Structured + BERT model achieved the highest ROC-AUC (0.6852), but its
PR-AUC improvement over the structured baseline was smaller and its Brier score
increased to 0.1269. Therefore, BERT did not consistently outperform TF-IDF
across the evaluated metrics.

Overall, the results suggest that textual borrower information contributes
incremental predictive value, but the magnitude of improvement over structured
credit information is modest. Structured + TF-IDF was retained as the selected
hybrid specification because it provided the most balanced performance across
ROC-AUC, PR-AUC, and Brier score.

These findings do not establish that TF-IDF is universally superior to BERT;
rather, they reflect the relative performance of the evaluated representations
under the experimental design used in this study.


## 7. Final Research Summary and Artifact Export

This section consolidates the principal outputs of the final evaluation.
The selected Structured + TF-IDF specification is summarized together with
its held-out predictive performance, calibration characteristics, incremental
value over the structured baseline, and validation-derived risk stratification.

The resulting artifacts are exported for use in the final dissertation analysis
and reporting. No further model selection or threshold optimization is performed
using the held-out test cohort.

### 7.1 Consolidated Final Model Summary

The final summary is assembled directly from the persisted comparison,
calibration, and ablation results to avoid manually duplicating metric values.


In [44]:
selected_representation = representation_comparison[
    representation_comparison["representation"] == SELECTED_MODEL
].iloc[0]

selected_calibration_row = selected_calibration.iloc[0]

selected_ablation_row = hybrid_ablation[
    hybrid_ablation["representation"] == SELECTED_MODEL
].iloc[0]

final_model_summary = pd.DataFrame([{
    "selected_model": SELECTED_MODEL,
    "test_roc_auc": selected_representation["roc_auc"],
    "test_pr_auc": selected_representation["pr_auc"],
    "test_brier_score": selected_representation["brier_score"],
    "test_ece_10_bins": selected_calibration_row["ece_10_bins"],
    "mean_predicted_probability":
        selected_calibration_row["mean_predicted_probability"],
    "observed_default_rate":
        selected_calibration_row["observed_default_rate"],
    "delta_roc_auc_vs_structured":
        selected_ablation_row["delta_roc_auc"],
    "delta_pr_auc_vs_structured":
        selected_ablation_row["delta_pr_auc"],
    "brier_improvement_vs_structured":
        selected_ablation_row["brier_improvement"],
    "approve_review_threshold": LOW_RISK_THRESHOLD,
    "review_reject_threshold": HIGH_RISK_THRESHOLD
}])

final_model_summary.round(4)


,selected_model,test_roc_auc,test_pr_auc,test_brier_score,test_ece_10_bins,mean_predicted_probability,observed_default_rate,delta_roc_auc_vs_structured,delta_pr_auc_vs_structured,brier_improvement_vs_structured,approve_review_threshold,review_reject_threshold
0,Structured + TF-IDF,0.6834,0.2788,0.1261,0.0089,0.1639,0.1585,0.0054,0.0031,0.0004,0.1372,0.3073


### 7.2 Persist Thesis-Ready Artifacts

The consolidated model summary, risk-band results, and hybrid ablation table are
saved to the Notebook 06 output directory for use in the dissertation.


In [45]:
final_model_summary.to_csv(
    NB6_OUTPUT_DIR / "final_model_summary.csv",
    index=False
)

decision_summary.to_csv(
    NB6_OUTPUT_DIR / "risk_band_summary.csv",
    index=False
)

hybrid_ablation.to_csv(
    NB6_OUTPUT_DIR / "hybrid_ablation_summary.csv",
    index=False
)

print("Final artifacts saved to:")
print(NB6_OUTPUT_DIR)

print("\nFiles:")
for path in sorted(NB6_OUTPUT_DIR.iterdir()):
    print("-", path.name)


Final artifacts saved to:
/content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_06

Files:
- final_model_summary.csv
- hybrid_ablation_summary.csv
- model_selection_summary.csv
- risk_band_summary.csv
- selected_model_summary.csv


### 7.3 Final Reproducibility Checks

Final assertions verify test-cohort alignment, monotonic risk-band ordering,
non-missing evaluation metrics, and valid threshold ordering before the
notebook is closed.


In [46]:
assert len(decision_results) == 15175

assert (
    decision_results["observed_default"].mean()
    == y_text_test.mean()
)

risk_rates = (
    decision_summary
    .set_index("decision")
    ["observed_default_rate"]
)

assert (
    risk_rates["Approve"]
    < risk_rates["Review"]
    < risk_rates["Reject"]
)

assert final_model_summary[
    [
        "test_roc_auc",
        "test_pr_auc",
        "test_brier_score",
        "test_ece_10_bins"
    ]
].notna().all().all()

assert (
    LOW_RISK_THRESHOLD
    < HIGH_RISK_THRESHOLD
)

print("All final research checks passed.")

All final research checks passed.


## 8. Notebook Summary and Handoff

This notebook completed the final analytical stage of the study.

The Structured + TF-IDF Logistic Regression model was retained as the final
research specification because it provided the strongest overall balance of
discrimination, probability accuracy, calibration, and interpretability among
the evaluated hybrid models.

Calibration assessment showed a small difference between mean predicted and
observed default risk, with a Brier score of 0.1261 and 10-bin ECE of 0.0089.
No additional post-hoc recalibration was therefore applied.

Validation-derived probability thresholds produced clear temporal risk
stratification on the held-out 2014 cohort, with observed default rates of
8.34%, 20.02%, and 34.21% across the lower-, intermediate-, and higher-risk
groups respectively.

Ablation analysis showed that textual borrower information provided modest
incremental predictive value beyond structured credit variables. BERT achieved
the highest ROC-AUC, while TF-IDF produced the more balanced hybrid performance
across ROC-AUC, PR-AUC, and Brier score.

The final tables and decision-support summaries have been persisted for use in
the dissertation results and discussion chapters. No further predictive model
development is required in this notebook.
